In [ ]:
!pip install sentence-transformers pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

In [4]:
df = pd.read_csv("/content/sample_data/ecommerce_clothing_complaints.csv")

print(df.head())
print(df.shape)

                                      complaint_text  complaint_type language  \
0  Saree color was red online but received pink, ...  color_mismatch  english   
1    Fabric is rough and fake, returning immediately     fake_fabric  english   
2  Ordered XL kurta but received M size, totally ...      wrong_size  english   
3  Ordered XL kurta but received M size, totally ...      wrong_size  english   
4  Saree color was red online but received pink, ...  color_mismatch  english   

   anger_score  days_pending  repeat_count  escalation_label  
0         0.57             9             4                 1  
1         0.58             1             1                 0  
2         0.54             7             1                 0  
3         0.68             4             3                 1  
4         0.79             9             3                 1  
(1000, 7)


In [5]:
print(df.columns)

print(df.isnull().sum())

Index(['complaint_text', 'complaint_type', 'language', 'anger_score',
       'days_pending', 'repeat_count', 'escalation_label'],
      dtype='object')
complaint_text      0
complaint_type      0
language            0
anger_score         0
days_pending        0
repeat_count        0
escalation_label    0
dtype: int64


In [6]:
df["complaint_text"] = (
    df["complaint_text"]
    .str.lower()
    .str.strip()
)

In [7]:
df.to_csv(
    "cleaned_complaints.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [8]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
complaints = df["complaint_text"].tolist()

embeddings = model.encode(
    complaints,
    show_progress_bar=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [10]:
embedding_df = pd.DataFrame(embeddings)

embedding_df.head()

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,-0.078707,-0.405002,-0.359498,0.057333,-0.047520,0.058526,0.355419,-0.268202,0.151251,-0.032372,...,-0.233797,-0.635402,0.011471,-0.033053,-0.050783,0.248939,0.297170,-0.325869,-0.286672,0.360135
1,-0.219531,-0.076064,0.195096,0.331476,0.056774,-0.126347,0.017351,-0.009357,-0.671461,0.414007,...,-0.207629,-0.603962,-0.361996,0.202588,0.143694,0.397318,0.032746,-0.428964,0.297488,0.146271
2,0.042451,0.273864,-0.095832,0.101611,-0.068555,-0.169342,-0.164078,0.196184,0.131290,-0.280336,...,0.132507,-0.509482,-0.524378,-0.189470,0.009803,-0.106200,-0.152106,-0.283200,-0.454764,0.256071
3,0.042451,0.273864,-0.095832,0.101611,-0.068555,-0.169342,-0.164078,0.196184,0.131290,-0.280336,...,0.132507,-0.509482,-0.524378,-0.189470,0.009803,-0.106200,-0.152106,-0.283200,-0.454764,0.256071
4,-0.078707,-0.405002,-0.359498,0.057333,-0.047520,0.058526,0.355419,-0.268202,0.151251,-0.032372,...,-0.233797,-0.635402,0.011471,-0.033053,-0.050783,0.248939,0.297170,-0.325869,-0.286672,0.360135


In [11]:
embedding_df.to_csv(
    "complaint_embeddings.csv",
    index=False
)

print("Embeddings Saved")

Embeddings Saved


In [12]:
numerical_features = df[
    [
        "anger_score",
        "days_pending",
        "repeat_count"
    ]
]

feature_df = pd.concat(
    [
        df[
            [
                "complaint_type",
                "escalation_label"
            ]
        ].reset_index(drop=True),

        numerical_features.reset_index(drop=True),

        embedding_df.reset_index(drop=True)
    ],
    axis=1
)

In [13]:
print(feature_df.shape)

(1000, 389)


In [14]:
feature_df.to_csv(
    "features.csv",
    index=False
)

print("Feature Matrix Saved")

Feature Matrix Saved


In [ ]:
from google.colab import files

files.download("cleaned_complaints.csv")
files.download("complaint_embeddings.csv")
files.download("features.csv")